# Stage 3 — Tracked 2D WholeBody Pose QA

This notebook consumes the frozen Stage-2 handoff, reuses the RTMW WholeBody133 cache, and exports `TrackedPose2DState`. It does not perform team, world-coordinate, legal-body or offside reasoning.


In [3]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd()
STAGE2_DIR = Path(r"D:\NCKH\stage2_sst_rtmw_v1.3\runs\stage2")
OUTPUT_DIR = PROJECT_DIR / "runs" / "stage3_v01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_DIR, STAGE2_DIR, OUTPUT_DIR, sep="\n")


d:\NCKH\stage3_pose2d_v0.1
D:\NCKH\stage2_sst_rtmw_v1.3\runs\stage2
d:\NCKH\stage3_pose2d_v0.1\runs\stage3_v01


## 1. Preflight

The structural gate checks the three Stage-2 JSON files, raw-pixel coordinate space, candidate IDs, t0 bboxes, and RTMW cache uniqueness. Missing RTMW pose evidence is reported explicitly.


In [4]:
from stage3_pose2d.stage2_adapter import load_stage2_bundle

bundle = load_stage2_bundle(stage2_dir=STAGE2_DIR, strict=True)
print(json.dumps(bundle.validation, indent=2, ensure_ascii=False))


{
  "ready": true,
  "errors": [],
  "warnings": [],
  "candidate_tracks": 10,
  "missing_pose_at_selected_frame": [],
  "ambiguous_track_frame_pose_mapping": [],
  "invalid_wholebody_records": [],
  "score_semantics": "RTMW_RAW_SIMCC_MAX_NOT_CALIBRATED_PROBABILITY",
  "coordinate_space": "RAW_DISTORTED_PIXEL"
}


## 2. Cache-only Stage 3 run

This is the preferred mode. RTMW is **not** run again.


In [5]:
from stage3_pose2d import Stage3Config, run_stage3

state = run_stage3(
    stage2_dir=STAGE2_DIR,
    output_dir=OUTPUT_DIR,
    config=Stage3Config(
        emit_temporal_estimates=False,
        enable_fallback_reinference=False,
    ),
)
print(json.dumps(state["metrics"], indent=2, ensure_ascii=False))


{
  "candidate_tracks": 10,
  "PoseCoverageAtT0_given_stage2_candidate": 1.0,
  "AcceptedPoseCoverageAtT0_given_stage2_candidate": 1.0,
  "ValidPoseCoverageAtT0_given_stage2_candidate": 1.0,
  "FootPoseCoverageAtT0_given_stage2_candidate": 1.0,
  "selected_frame_status_counts": {
    "VALID": 10,
    "DEGRADED": 0,
    "REJECTED": 0,
    "MISSING": 0
  },
  "upstream_stage2_candidate_recall_not_recomputed": true
}


## 3. Inspect selected-frame quality

Raw RTMW `raw_model_score` is SimCC evidence, not probability. Stage 3 attaches independent QA states.


In [6]:
for tr in state["tracks"]:
    t0_obs = next((o for o in tr["observations"] if o["frame_index"] == state["replay_context"]["selected_frame"]), None)
    print(tr["track_id"], tr["upstream_role"], tr["selected_frame_pose_status"], None if t0_obs is None else t0_obs.get("qa"))


track_001 player VALID {'pose_status': 'VALID', 'body_completeness': 1.0, 'feet_completeness': 1.0, 'core_completeness': 1.0, 'inside_fraction': 1.0, 'median_positive_raw_score': 3.2904438972473145, 'relative_low_evidence_floor': 1.1516553640365599, 'low_model_evidence_fraction': 0.0, 'bone_outlier_fraction': 0.0, 'bone_outliers': [], 'geometry_valid': True, 'note': 'raw_model_score is RTMW SimCC evidence, not a calibrated probability'}
track_002 player VALID {'pose_status': 'VALID', 'body_completeness': 1.0, 'feet_completeness': 1.0, 'core_completeness': 1.0, 'inside_fraction': 1.0, 'median_positive_raw_score': 3.572943925857544, 'relative_low_evidence_floor': 1.2505303740501403, 'low_model_evidence_fraction': 0.09022556390977443, 'bone_outlier_fraction': 0.0, 'bone_outliers': [], 'geometry_valid': True, 'note': 'raw_model_score is RTMW SimCC evidence, not a calibrated probability'}
track_003 player VALID {'pose_status': 'VALID', 'body_completeness': 1.0, 'feet_completeness': 1.0, 'co

## 4. Optional QA image

If Stage 2 still points to an accessible source video, Stage 3 writes `selected_frame_pose_qa.png`.


In [7]:
from IPython.display import Image, display
img = state.get("artifacts", {}).get("selected_frame_pose_qa")
if img and Path(img).is_file():
    display(Image(filename=img))
else:
    print("No portable source video / overlay not generated.")


No portable source video / overlay not generated.


## 5. Optional controlled fallback

Enable only after inspecting failures. The same RTMW-L model is tried at crop scales 1.0/1.1/1.2. If fallback wins, the original Stage-2 pose remains under `upstream_raw_pose`.


In [8]:
# Example only — uncomment when needed.
# state_fb = run_stage3(
#     stage2_dir=STAGE2_DIR,
#     output_dir=PROJECT_DIR / "runs" / "stage3_v01_fallback",
#     config=Stage3Config(
#         enable_fallback_reinference=True,
#         rtmw_model=str(PROJECT_DIR / "weights" / "rtmw_l_384x288.onnx"),
#         fallback_crop_scales=[1.0, 1.1, 1.2],
#     ),
# )


## 6. Football-domain benchmark

After downloading public 3DSP and the RTMW-L ONNX, run from a terminal:

```powershell
python benchmark_stage3.py inspect-3dsp --root .\data\3dsp
python benchmark_stage3.py run-3dsp --root .\data\3dsp --split train --rtmw-model .\weights\rtmw_l_384x288.onnx --output-dir .\benchmark_results\3dsp_rtmw_l
```
